# Hachi conversation improvement — pilot, resume, export
Continues your **registered hachi-master** weights. Creators: **Axeil Escabal and Beomarc Cartoneros**.

Enable a GPU and Internet in Kaggle. Attach the input package and `hachi-master-current.gguf`.
The first run deliberately pauses after **2 optimizer steps**. Download the recovery ZIP,
reattach it with the same original GGUF in a fresh session, and set `START_NEW_RUN=False`.
Keep `PILOT_STEPS=2` for one resume test; set it to `0` once both tests pass.

Use **Save Version → Save & Run All** to produce saved notebook output. A local autosave
does not survive every forced shutdown: download successful saved output before ending a session.
Setup/training errors are shown below and recorded in `hachi-session-status.json`; a green
notebook version alone does not mean training succeeded. See `START_HERE.md` for full instructions.


In [ ]:
import time
SESSION_START = time.time()
START_NEW_RUN = True
PILOT_STEPS = 2          # Number of NEW optimizer steps this session; 0 runs the remaining schedule.
SESSION_HOURS = 8        # Budget measured from this cell; training reserves the final 20 minutes.
EXPORT_RESULT = True    # After the full training schedule, merge and create an Ollama GGUF.
EXPORT_ONLY = False     # Use a completed recovery to retry conversion without training again.
RESUME_FROM = ""        # Optional exact /kaggle/input/... recovery ZIP or extracted folder.
PACKAGE_FROM = ""       # Optional exact initial ZIP or extracted package folder (fresh runs only).
SOURCE_FROM = ""        # Optional exact path to hachi-master-current.gguf.


In [ ]:
import json
from pathlib import Path
import subprocess
import sys
if not Path('/kaggle/input').is_dir():
    raise RuntimeError('This notebook is intended for Kaggle; training is not started locally.')
bootstrap = Path('/kaggle/working/hachi-bootstrap')
bootstrap.mkdir(exist_ok=True)
embedded_sources = {'recovery.py': '"""Atomic, checksummed recovery bundles. This module has no ML dependencies."""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path, PurePosixPath\nimport shutil\nimport stat\nimport zipfile\n\nSCHEMA = "hachi-conversation-recovery-v1"\nREQUIRED = {"adapter_config.json", "adapter_model.safetensors", "trainer_state.json",\n            "optimizer.pt", "scheduler.pt", "rng_state.pth", "scaler.pt"}\n\n\ndef digest(path):\n    h = hashlib.sha256()\n    with Path(path).open("rb") as f:\n        for block in iter(lambda: f.read(1024 * 1024), b""):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef atomic_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(path.suffix + ".tmp")\n    with tmp.open("w", encoding="utf-8") as f:\n        json.dump(value, f, indent=2, ensure_ascii=False)\n        f.flush()\n        os.fsync(f.fileno())\n    os.replace(tmp, path)\n\n\ndef checkpoint_step(checkpoint):\n    checkpoint = Path(checkpoint)\n    missing = [name for name in REQUIRED if not (checkpoint / name).is_file() or (checkpoint / name).stat().st_size == 0]\n    if missing:\n        raise ValueError("Incomplete training checkpoint: " + ", ".join(sorted(missing)))\n    step = json.loads((checkpoint / "trainer_state.json").read_text(encoding="utf-8"))["global_step"]\n    if type(step) is not int or step < 1 or checkpoint.name != f"checkpoint-{step}":\n        raise ValueError("Checkpoint directory and training step disagree")\n    return step\n\n\ndef latest_complete(run):\n    candidates = []\n    for path in Path(run).glob("checkpoint-*"):\n        if path.is_dir():\n            try:\n                candidates.append((checkpoint_step(path), path))\n            except (ValueError, KeyError, json.JSONDecodeError):\n                pass\n    return max(candidates, default=(0, None))[1]\n\n\ndef publish_recovery(checkpoint, run, destination):\n    """Publish only a complete checkpoint; failed writes preserve the prior ZIP."""\n    checkpoint, run, destination = Path(checkpoint), Path(run), Path(destination)\n    step = checkpoint_step(checkpoint)\n    identity = json.loads((run / "run-identity.json").read_text(encoding="utf-8"))\n    files = {f"checkpoint/{p.relative_to(checkpoint).as_posix()}": p\n             for p in checkpoint.rglob("*") if p.is_file()}\n    for p in run.joinpath("package").rglob("*"):\n        if p.is_file() and "__pycache__" not in p.parts:\n            files[f"package/{p.relative_to(run / \'package\').as_posix()}"] = p\n    files["run-identity.json"] = run / "run-identity.json"\n    for name in ("requirements.lock.txt", "run-summary.json", "baseline.json"):\n        if (run / name).is_file():\n            files[name] = run / name\n    manifest = {"schema": SCHEMA, "step": step, "identity": identity,\n                "files": {name: {"sha256": digest(p), "bytes": p.stat().st_size} for name, p in files.items()},\n                "base_weights_included": False,\n                "note": "Reattach the original Hachi weights; their SHA-256 must match run-identity.json."}\n    destination.parent.mkdir(parents=True, exist_ok=True)\n    tmp = destination.with_suffix(".zip.tmp")\n    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED, compresslevel=1) as z:\n        for name, p in files.items():\n            if p.is_symlink():\n                raise ValueError("Refusing symlink in recovery")\n            z.write(p, name)\n        z.writestr("recovery-info.json", json.dumps(manifest, indent=2))\n    with zipfile.ZipFile(tmp) as z:\n        if z.testzip() is not None:\n            raise ValueError("Recovery ZIP failed CRC validation")\n    with tmp.open("r+b") as f:\n        os.fsync(f.fileno())\n    # The last known-good archive is retained independently of checkpoint pruning.\n    if destination.exists():\n        previous = destination.with_name(destination.stem + "-previous.zip")\n        shutil.copyfile(destination, previous.with_suffix(".zip.tmp"))\n        os.replace(previous.with_suffix(".zip.tmp"), previous)\n    os.replace(tmp, destination)\n    print(f"BACKUP READY: optimizer step {step}; {destination}", flush=True)\n    return manifest\n\n\ndef safe_extract(archive, destination, max_bytes=8 * 1024**3):\n    destination = Path(destination).resolve()\n    with zipfile.ZipFile(archive) as z:\n        seen = set()\n        total = 0\n        for item in z.infolist():\n            # ZipInfo normalizes backslashes on Windows; inspect its original\n            # member name so validation behaves the same on Kaggle and Windows.\n            name = item.orig_filename\n            path = PurePosixPath(name)\n            total += item.file_size\n            if (name in seen or "\\x00" in name or "\\\\" in name or ":" in name or path.is_absolute()\n                    or ".." in path.parts or stat.S_ISLNK(item.external_attr >> 16)\n                    or not (destination / name).resolve().is_relative_to(destination)):\n                raise ValueError("Unsafe archive entry: " + name)\n            if total > max_bytes:\n                raise ValueError("Archive exceeds recovery size limit")\n            seen.add(name)\n        if z.testzip() is not None:\n            raise ValueError("Corrupt archive")\n        destination.mkdir(parents=True, exist_ok=True)\n        z.extractall(destination)\n\n\ndef validate_recovery(folder, expected_identity=None):\n    folder = Path(folder).resolve()\n    info = json.loads((folder / "recovery-info.json").read_text(encoding="utf-8"))\n    if info.get("schema") != SCHEMA:\n        raise ValueError("Wrong Hachi recovery version")\n    if expected_identity is not None and info["identity"] != expected_identity:\n        raise ValueError("Run configuration, source weights or dataset changed; refusing resume")\n    for name, record in info["files"].items():\n        if ".." in PurePosixPath(name).parts or "\\\\" in name or ":" in name:\n            raise ValueError("Unsafe manifest path")\n        p = (folder / name).resolve()\n        if not p.is_relative_to(folder) or p.is_symlink() or not p.is_file():\n            raise ValueError("Missing or unsafe recovery asset: " + name)\n        if p.stat().st_size != record["bytes"] or digest(p) != record["sha256"]:\n            raise ValueError("Recovery checksum mismatch: " + name)\n    identity = json.loads((folder / "run-identity.json").read_text(encoding="utf-8"))\n    if identity != info["identity"]:\n        raise ValueError("Recovery identity mismatch")\n    for name in ("run-identity.json", "requirements.lock.txt", "package/package-manifest.json"):\n        if name not in info["files"]:\n            raise ValueError("Recovery missing required run asset: " + name)\n    if digest(folder / "package/package-manifest.json") != identity["package_sha256"]:\n        raise ValueError("Recovery package identity mismatch")\n    for name in REQUIRED:\n        if "checkpoint/" + name not in info["files"]:\n            raise ValueError("Recovery missing required checkpoint asset: " + name)\n    state = json.loads((folder / "checkpoint/trainer_state.json").read_text(encoding="utf-8"))\n    if type(info["step"]) is not int or info["step"] < 1 or state["global_step"] != info["step"]:\n        raise ValueError("Recovery step mismatch")\n    return info\n\n\ndef restore_checkpoint(folder, run, expected_identity):\n    folder, run = Path(folder), Path(run)\n    info = validate_recovery(folder, expected_identity)\n    checkpoint = run / f"checkpoint-{info[\'step\']}"\n    if checkpoint.exists():\n        raise ValueError("Restore destination exists; use a fresh run folder")\n    shutil.copytree(folder / "checkpoint", checkpoint)\n    checkpoint_step(checkpoint)\n    return checkpoint\n', 'session.py': '"""Notebook subprocess driver. Setup failures leave prior recovery downloadable."""\nimport argparse\nimport importlib.metadata\nimport json\nfrom pathlib import Path\nimport shutil\nimport subprocess\nimport sys\nimport time\nimport traceback\n\nfrom recovery import atomic_json, digest, latest_complete, publish_recovery, safe_extract, validate_recovery\n\n\ndef one(paths, description):\n    paths = sorted(set(Path(p).resolve() for p in paths))\n    if len(paths) != 1:\n        raise ValueError(f"Expected exactly one {description}, found {len(paths)}: {paths}. Set the explicit path in cell 1.")\n    return paths[0]\n\n\ndef run_session(settings):\n    inputs = Path("/kaggle/input")\n    working = Path("/kaggle/working")\n    stage = working / ".hachi-stage"\n    run = working / "hachi-run"\n    backup = working / "hachi-conversation-v1-recovery.zip"\n    if stage.exists() or run.exists():\n        raise ValueError("This session already has Hachi output. Download it and start a fresh session before rerunning.")\n    stage.mkdir()\n    deadline = settings["session_start"] + settings["session_hours"] * 3600\n    if not 0 < settings["session_hours"] <= 8:\n        raise ValueError("Use a session budget greater than 0 and at most 8 hours")\n\n    def command(argv, reserve_seconds=120):\n        remaining = deadline - time.time() - reserve_seconds\n        if remaining <= 0:\n            raise TimeoutError("Session budget exhausted; keep the last complete recovery ZIP")\n        subprocess.run([str(x) for x in argv], check=True, timeout=remaining)\n\n    attached = list(inputs.rglob("hachi-conversation-v1-recovery.zip"))\n    attached += [p.parent for p in inputs.rglob("recovery-info.json")]\n    resume = None\n    if settings["start_new_run"]:\n        if attached or settings["resume_from"] or settings["export_only"]:\n            raise ValueError("Fresh mode conflicts with attached recovery or export-only. Set START_NEW_RUN=False to resume.")\n        supplied = settings["package_from"]\n        candidates = [Path(supplied)] if supplied else list(inputs.rglob("hachi-conversation-v1-input.zip"))\n        if not candidates:\n            candidates = [p.parent for p in inputs.rglob("package-manifest.json")]\n        chosen = one(candidates, "input package")\n        if chosen.is_file():\n            package = stage / "package"\n            safe_extract(chosen, package)\n        else:\n            package = chosen\n    else:\n        chosen = one([settings["resume_from"]] if settings["resume_from"] else attached, "recovery ZIP or folder")\n        if chosen.is_file():\n            resume = stage / "recovery"\n            safe_extract(chosen, resume)\n        else:\n            resume = chosen\n        validate_recovery(resume)\n        package = resume / "package"\n        # Publish an independent copy before dependency installation or GPU setup.\n        if chosen.is_file():\n            shutil.copyfile(chosen, backup)\n        else:\n            import zipfile\n            with zipfile.ZipFile(backup, "w", zipfile.ZIP_DEFLATED, compresslevel=1) as z:\n                for item in resume.rglob("*"):\n                    if item.is_file():\n                        z.write(item, item.relative_to(resume).as_posix())\n\n    # Verify every packaged source before importing it or installing its requirements.\n    manifest = json.loads((package / "package-manifest.json").read_text())\n    if manifest.get("schema") != "hachi-conversation-package-v1":\n        raise ValueError("Wrong package version")\n    for name, sha in manifest["files"].items():\n        item = (package / name).resolve()\n        if not item.is_relative_to(package.resolve()) or digest(item) != sha:\n            raise ValueError("Package checksum mismatch: " + name)\n    source_info = json.loads((package / "source_model.json").read_text())\n    source = one([settings["source_from"]] if settings["source_from"] else inputs.rglob(source_info["filename"]), "original Hachi GGUF")\n    if digest(source) != source_info["sha256"]:\n        raise ValueError("Wrong base weights; reattach hachi-master-current.gguf from this package")\n    requirements = resume / "requirements.lock.txt" if resume else package / "requirements.txt"\n    if not requirements.is_file():\n        raise ValueError("Missing dependency lock; cannot safely resume")\n    command([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "-r", requirements])\n    torch_version = importlib.metadata.version("torch")\n    if tuple(int(n) for n in torch_version.split("+")[0].split(".")[:2]) < (2, 6):\n        raise RuntimeError("Select a Kaggle GPU image with CUDA PyTorch >=2.6")\n    export_only = settings["export_only"]\n    if resume:\n        state = json.loads((resume / "checkpoint/trainer_state.json").read_text())\n        if state["global_step"] >= state["max_steps"]:\n            export_only = True\n            print("Checkpoint has finished the training schedule; continuing directly to export.", flush=True)\n    if not export_only:\n        argv = [sys.executable, package / "train.py", "--package", package, "--source", source,\n                "--run", run, "--deadline", deadline, "--pilot-steps", settings["pilot_steps"], "--reserve-minutes", 20]\n        if resume:\n            argv += ["--resume", resume]\n        command(argv)\n        summary = json.loads((run / "run-summary.json").read_text())\n        if summary["status"] != "training_complete":\n            print("PILOT/SESSION PAUSED. Download recovery ZIP and use resume mode in a fresh session.", flush=True)\n            return\n    if settings["export_result"] or export_only:\n        export_recovery = stage / "export-recovery"\n        safe_extract(backup, export_recovery)\n        command([sys.executable, package / "export.py", "--recovery", export_recovery,\n                 "--source", source, "--output", working / "hachi-export", "--deadline", deadline])\n    else:\n        print("Training complete. Download recovery; enable EXPORT_ONLY in a fresh session to convert.", flush=True)\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--settings", required=True)\n    args = parser.parse_args()\n    settings = json.loads(args.settings)\n    status = {"status": "started"}\n    try:\n        run_session(settings)\n        status = {"status": "finished_without_error", "note": "Read hachi-run/run-summary.json for paused versus complete."}\n    except BaseException as exc:\n        traceback.print_exc()\n        status = {"status": "failed", "error": str(exc)}\n        run = Path("/kaggle/working/hachi-run")\n        try:\n            checkpoint = latest_complete(run)\n            if checkpoint and (run / "run-identity.json").is_file():\n                publish_recovery(checkpoint, run, run.parent / "hachi-conversation-v1-recovery.zip")\n        except Exception:\n            traceback.print_exc()\n        print("RUN FAILED. Any existing recovery ZIP is the last complete state, not proof of training success.", flush=True)\n    finally:\n        atomic_json(Path("/kaggle/working/hachi-session-status.json"), status)\n    return 1 if status["status"] == "failed" else 0\n\n\nif __name__ == "__main__":\n    sys.exit(main())\n'}
for name, source in embedded_sources.items():
    (bootstrap / name).write_text(source, encoding='utf-8')
settings = dict(session_start=SESSION_START, start_new_run=START_NEW_RUN,
    pilot_steps=PILOT_STEPS, session_hours=SESSION_HOURS, export_result=EXPORT_RESULT,
    export_only=EXPORT_ONLY, resume_from=RESUME_FROM, package_from=PACKAGE_FROM, source_from=SOURCE_FROM)
completed = subprocess.run([sys.executable, str(bootstrap / 'session.py'), '--settings', json.dumps(settings)])
print('Driver exit code:', completed.returncode)
if completed.returncode:
    print('FAILED: inspect the error above. The download cell can still expose the last valid backup.')


In [ ]:
from IPython.display import display, FileLink
import json
from pathlib import Path
working = Path('/kaggle/working')
for name in ('hachi-session-status.json', 'hachi-run/run-summary.json'):
    p = working / name
    if p.is_file():
        print(name, p.read_text())
for name in ('hachi-conversation-v1-recovery.zip', 'hachi-conversation-v1-recovery-previous.zip', 'hachi-conversation-v1-result.zip'):
    p = working / name
    if p.is_file():
        print(f'{name}: {p.stat().st_size / 1024**2:.1f} MiB')
        display(FileLink(str(p)))
if not (working / 'hachi-conversation-v1-recovery.zip').exists():
    print('NO RECOVERY YET: no complete optimizer checkpoint has been published.')
print('Download saved notebook output. Resume needs the recovery ZIP plus the SAME original Hachi GGUF.')
